# 06 · Structure and execution embeddings
### How to turn symbolic programs and their behavior into useful vectors

**Question:** How can the neural policy from lesson 05 reason about a new program tree, rather than relying on a fixed set of hand-engineered features?

We will build a tiny **executable grammar**, inspect its **syntax trees**, create three program representations, train a neural model that observes the structure and the input, and test whether the resulting vectors tell us something useful:

1. **Operator bag:** what operations occur, but not how they are connected.
2. **Recursive structural embedding:** a compositional vector that remembers child order; initially random and **not trained**.
3. **Execution embedding:** the behavior of a program on a chosen set of probe inputs, compressed into a vector.

We then learn a program-conditioned neural representation by **predicting the executed output from the tree's positional encoding and an input mask**. This is a small *supervised neural execution emulator*, not a general graph neural network, not a full Tree-LSTM, and not a solver for unseen ARC tasks.

**Learning route:** AST → typed operations → representation collisions → tree encoders → semantic signatures → finite-probe ambiguity → learned embeddings → separate program/input holdouts → retrieval & caching → interface to Grammar-UCT.

**Prerequisites:** 00–05 (especially 02 and 05). NumPy and Matplotlib only. Every diagram uses synthetic 1D binary grids of width seven. Run cells top to bottom. Prediction prompts appear before the corresponding experiments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from collections import defaultdict
from itertools import product

SEED = 606
WIDTH = 7
RNG = np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize': (9, 3.6), 'axes.grid': True, 'grid.alpha': .17})
np.set_printoptions(precision=3, suppress=True)
print("Notebook 06: pure NumPy + Matplotlib, deterministic seeds.")

## 1 · A program is a tree, not an arbitrary vector

Our tiny grammar operates on Boolean rows, with two leaves, four unary operations, and two binary operations:

| Operator | Inputs | Output | Meaning |
|---|---|---|---|
| INPUT | – | Mask | Current input |
| ZERO | – | Mask | All-false mask |
| RIGHT | Mask | Mask | Shift right one, discarding the last cell |
| LEFT | Mask | Mask | Shift left one, discarding the first cell |
| FLIP | Mask | Mask | Reverse cell order |
| NOT | Mask | Mask | Boolean complement |
| OR | Mask, Mask | Mask | Union |
| AND | Mask, Mask | Mask | Intersection |

A node includes its operator **and ordered children**. A legal AST is an executable hypothesis. Unlike a sequence of free-text labels, the tree can be checked against operator arity and typed before execution.

**Predict:** Is FLIP(RIGHT(INPUT)) always equivalent to RIGHT(FLIP(INPUT))? What information would be lost if both were represented solely by the counts {FLIP:1, RIGHT:1}?

In [ ]:
OPS = ("INPUT", "ZERO", "RIGHT", "LEFT", "FLIP", "NOT", "OR", "AND")
ARITY = {"INPUT": 0, "ZERO": 0, "RIGHT": 1, "LEFT": 1, "FLIP": 1,
         "NOT": 1, "OR": 2, "AND": 2}

@dataclass(frozen=True)
class Expr:
    op: str
    children: tuple = ()
    def __post_init__(self):
        if self.op not in ARITY or len(self.children) != ARITY[self.op]:
            raise ValueError("Invalid operator or arity")
        if any(not isinstance(c, Expr) for c in self.children):
            raise TypeError("Children must be Expr instances")

X = Expr("INPUT")
Z = Expr("ZERO")
def U(op, child): return Expr(op, (child,))
def B(op, left, right): return Expr(op, (left, right))
def pretty(e):
    return e.op if not e.children else e.op + "(" + ", ".join(map(pretty, e.children)) + ")"
def size(e): return 1 + sum(size(c) for c in e.children)
def depth(e): return 0 if not e.children else 1 + max(depth(c) for c in e.children)

p1 = U("FLIP", U("RIGHT", X))
p2 = U("RIGHT", U("FLIP", X))
p3 = U("FLIP", U("FLIP", X))
p4 = B("OR", U("RIGHT", X), U("FLIP", X))
p5 = B("OR", U("FLIP", X), U("RIGHT", X))
examples = [X, U("RIGHT", X), p1, p2, p3, p4, p5]
print(*(pretty(p) for p in examples), sep="\n")
assert p1 != p2 and p4 != p5
try:
    Expr("OR", (X,))
    raise AssertionError("Invalid arity should be rejected.")
except ValueError:
    print("Arity validation: OK.")

## 2 · Visualize connections, not merely the vocabulary

Each parent combines **particular child results**. Unary composition means sequence order matters; binary operators introduce branches and potentially shared subexpressions.

The diagram below is a *syntax tree*. A DAG implementation may share repeated subtrees instead of storing separate copies; we will revisit caching later.

**Predict:** Which two trees have the same operator multiset yet a different computation order?

In [ ]:
def draw_tree(ax, expr, title):
    positions = {}
    leaf_counter = [0]
    def place(e, level=0):
        if not e.children:
            x = leaf_counter[0]
            leaf_counter[0] += 1
        else:
            child_x = [place(c, level + 1) for c in e.children]
            x = float(np.mean(child_x))
        positions[id(e)] = (x, -level, e)
        return x
    place(expr)
    for x, y, e in positions.values():
        for child in e.children:
            cx, cy, _ = positions[id(child)]
            ax.plot([x,cx],[y,cy],color="0.60",lw=1.5)
    for x,y,e in positions.values():
        ax.text(x,y,e.op,fontsize=8,ha="center",va="center",
                bbox=dict(boxstyle="round,pad=.27",fc="#dbeafe",ec="#6a7f9c"))
    ax.set_xlim(-.75,max(leaf_counter[0]-.25,.75))
    ax.set_ylim(-depth(expr)-.48,.5)
    ax.set_title(title)
    ax.axis("off")

fig,axs=plt.subplots(1,3,figsize=(11,4),layout="constrained")
for a,p,t in zip(axs,[p1,p2,p4],["FLIP after RIGHT","RIGHT after FLIP","Two-branch OR"]):
    draw_tree(a,p,t)
plt.show()

## 3 · Execute precisely, including boundary behavior

Represent each input row as seven Booleans. Every operation preserves the row length. RIGHT and LEFT lose information at the boundary; NOT complements every cell; binary OR/AND operate elementwise.

Our interpreter supports **batched rows** with shape (number of examples, width), so a program can be evaluated over many test inputs without a Python loop over rows.

**Predict:** If we reverse a row and then shift right, which original boundary cell disappears?

In [ ]:
def execute(expr, grids):
    g = np.asarray(grids, dtype=bool)
    if g.ndim not in (1, 2) or g.shape[-1] != WIDTH:
        raise ValueError("Expected one Boolean row or a batch of width-seven rows")
    if expr.op == "INPUT": return g.copy()
    if expr.op == "ZERO": return np.zeros_like(g)
    children = [execute(c,g) for c in expr.children]
    if expr.op == "RIGHT":
        out = np.zeros_like(children[0]);out[...,1:]=children[0][...,:-1];return out
    if expr.op == "LEFT":
        out = np.zeros_like(children[0]);out[...,:-1]=children[0][...,1:];return out
    if expr.op == "FLIP": return children[0][...,::-1].copy()
    if expr.op == "NOT": return ~children[0]
    if expr.op == "OR": return children[0] | children[1]
    if expr.op == "AND": return children[0] & children[1]
    raise ValueError(expr.op)

onehot = np.zeros(WIDTH,dtype=bool);onehot[1]=True
def show_row(ax, a, title):
    ax.imshow(np.asarray(a,dtype=int)[None,:],cmap="Greys",vmin=0,vmax=1,aspect="auto")
    ax.set(xticks=range(WIDTH),yticks=[],title=title)
fig,axs=plt.subplots(1,4,figsize=(12,1.9),layout="constrained")
for ax,(p,t) in zip(axs,[(X,"Input"),(p1,"FLIP(RIGHT)"),(p2,"RIGHT(FLIP)"),(p3,"FLIP(FLIP)")]):
    show_row(ax,execute(p,onehot),t)
plt.show()
assert not np.array_equal(execute(p1,onehot),execute(p2,onehot))
assert np.array_equal(execute(p3,onehot),onehot)

## 4 · The bag-of-operators baseline loses order

The simplest feature extractor counts how often each operation appears in the tree:

\[\phi_{\mathrm{bag}}(p)_k=\sum_{v\in p}\mathbf{1}[\mathrm{op}(v)=k].\]

This is cheap, interpretable, and entirely unable to distinguish trees with the same operator multiset. For a policy conditioned on a partial program, that can erase precisely the information needed to choose a good next operation.

**Predict:** Will a bag encoder assign identical vectors to OR(RIGHT(INPUT),FLIP(INPUT)) and OR(FLIP(INPUT),RIGHT(INPUT))? Are those programs behaviorally identical too?

In [ ]:
OP_INDEX = {op:i for i,op in enumerate(OPS)}
def bag(e):
    out = np.zeros(len(OPS),dtype=float)
    out[OP_INDEX[e.op]] += 1
    for child in e.children: out += bag(child)
    return out
assert np.array_equal(bag(p1),bag(p2))
assert np.array_equal(bag(p4),bag(p5))
print("Same syntax histogram, different order:",np.array_equal(bag(p1),bag(p2)))
fig,ax=plt.subplots(figsize=(9.2,3.5))
x=np.arange(len(OPS))
ax.bar(x-.2,bag(p1),width=.4,label="FLIP(RIGHT)")
ax.bar(x+.2,bag(p2),width=.4,label="RIGHT(FLIP)",alpha=.65)
ax.set(xticks=x,xticklabels=OPS,ylabel="Operator counts",title="Identical bags hide composition")
ax.tick_params(axis="x",rotation=35);ax.legend();plt.show()

## 5 · A structural embedding uses recursive message passing

An intuitive tree encoder combines information from each child, with **different matrices for child positions**:

\[h_v=\tanh(e_{\operatorname{op}(v)}+W_Lh_{\mathrm{left}(v)}+W_Rh_{\mathrm{right}(v)}).\]

For unary nodes we use the left slot; absent children contribute zero. Because \(W_L\) and \(W_R\) differ, swapping children can produce a different representation even if operator counts are equal.

This is a **single shared, randomly initialized recursive encoder**. At this stage it is *not learned*, and vector proximity should **not** be interpreted as behavioral equivalence. The benefit is the inductive bias: variable-size trees become fixed-length vectors using shared composition rules.

**Predict:** Would setting \(W_L=W_R\) make the encoder insensitive to swapping children of a binary commutative operator? Would it *also* collapse genuinely order-sensitive operators?

In [ ]:
D=10
gen=np.random.default_rng(SEED)
NODE_EMBED=gen.normal(0,.45,(len(OPS),D))
W_LEFT=gen.normal(0,.25,(D,D))
W_RIGHT=gen.normal(0,.25,(D,D))
def recursive_embed(e):
    h = NODE_EMBED[OP_INDEX[e.op]].copy()
    if len(e.children)>0: h += W_LEFT@recursive_embed(e.children[0])
    if len(e.children)>1: h += W_RIGHT@recursive_embed(e.children[1])
    return np.tanh(h)
def cosine_matrix(z):
    z=np.asarray(z,dtype=float)
    z=z/np.maximum(np.linalg.norm(z,axis=1,keepdims=True),1e-12)
    return z@z.T
sample=[X,U("RIGHT",X),p1,p2,p3,p4,p5]
labels=["X","R(X)","F(R(X))","R(F(X))","F(F(X))","OR(R,F)","OR(F,R)"]
bag_sim=cosine_matrix([bag(p) for p in sample])
recursive_sim=cosine_matrix([recursive_embed(p) for p in sample])
fig,axs=plt.subplots(1,2,figsize=(11,4),layout="constrained")
for ax,a,title in zip(axs,[bag_sim,recursive_sim],["Bag similarity","Random recursive similarity"]):
    im=ax.imshow(a,vmin=-1,vmax=1,cmap="coolwarm")
    ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels,title=title)
    ax.tick_params(axis="x",labelrotation=60,labelsize=8);ax.tick_params(axis="y",labelsize=8)
fig.colorbar(im,ax=axs.ravel().tolist(),shrink=.72)
plt.show()
print("Bag identical (p1/p2):",np.array_equal(bag(p1),bag(p2)))
print("Recursive identical (p1/p2):",np.array_equal(recursive_embed(p1),recursive_embed(p2)))
assert not np.array_equal(recursive_embed(p1),recursive_embed(p2))

## 6 · A syntax distance is not necessarily a behavior distance

Two syntactically distinct programs may compute **exactly the same function**:

- FLIP(FLIP(INPUT)) is INPUT.
- OR(a,b) and OR(b,a) return the same mask.
- OR(INPUT,ZERO) is INPUT.

Conversely, the same operator multiset can express **different functions** when composition is ordered. This is why learned structure embeddings and observed execution embeddings are complementary.

For width seven, there are exactly \(2^7=128\) possible binary inputs. We can execute our tiny programs on **all** of them, providing an exhaustive semantic oracle for *this fixed toy domain*. This is unusual; real ARC grids are far too diverse for full enumeration.

**Predict:** If two programs agree on 5 probes, can we conclude they agree on all 128 possible inputs?

In [ ]:
codes=np.arange(2**WIDTH,dtype=np.uint16)
ALL_INPUTS=((codes[:,None]>>np.arange(WIDTH)) & 1).astype(bool)
assert ALL_INPUTS.shape==(128,7)
def signature(expr,inputs):
    return execute(expr,inputs).astype(np.uint8).reshape(-1)
def semantic_same(a,b,inputs=ALL_INPUTS):
    return np.array_equal(signature(a,inputs),signature(b,inputs))
assert semantic_same(p3,X)
assert semantic_same(p4,p5)
assert semantic_same(B("OR",X,Z),X)
assert not semantic_same(p1,p2)
probe_small=ALL_INPUTS[[0]]
assert semantic_same(p1,p2,probe_small) and not semantic_same(p1,p2,ALL_INPUTS)
fig,ax=plt.subplots(2,3,figsize=(11,3.3),layout="constrained")
for r,input_id in enumerate([0,2]):
    source=ALL_INPUTS[input_id]
    for c,(p,title) in enumerate([(X,"INPUT"),(p1,"FLIP(RIGHT)"),(p2,"RIGHT(FLIP)")]):
        show_row(ax[r,c],execute(p,source),title+f" on probe {input_id}")
plt.show()
print("Agree on all-zero probe:",semantic_same(p1,p2,probe_small))
print("Agree on entire 128-input domain:",semantic_same(p1,p2))

## 7 · Execution signatures: fingerprints tied to a probe distribution

For probes \(x_1,\ldots,x_m\), define

\[\psi_{\mathrm{exec}}(p)=[p(x_1),p(x_2),\ldots,p(x_m)].\]

This vector encodes **observable behavior** rather than syntax. It is exact on its chosen probes and knows nothing about *unseen* inputs. Two different trees with identical probe outputs collapse to the same execution signature.

Probe selection matters. An all-zero probe cannot distinguish many shifting operations. A collection containing singletons and diverse patterns discriminates more functions.

**Predict:** Should the program's identity in the search graph be replaced by its probe fingerprint? What provenance information might be lost?

In [ ]:
PROBE_IDS=np.array([0,1,2,4,8,16,32,64,127,21,37,90],dtype=int)
PROBES=ALL_INPUTS[PROBE_IDS]
def probe_embedding(p): return signature(p,PROBES).astype(float)
exec_sim=cosine_matrix([probe_embedding(p) for p in sample])
fig,axs=plt.subplots(1,2,figsize=(11,4),layout="constrained")
axs[0].imshow(PROBES,cmap="Greys",vmin=0,vmax=1,aspect="auto",interpolation="nearest")
axs[0].set(xlabel="Position",ylabel="Probe",title="Twelve execution probes",
           xticks=range(WIDTH),yticks=range(len(PROBES)))
im=axs[1].imshow(exec_sim,cmap="coolwarm",vmin=-1,vmax=1)
axs[1].set(xticks=range(len(labels)),yticks=range(len(labels)),
           xticklabels=labels,yticklabels=labels,title="Probe-execution similarity")
axs[1].tick_params(axis="x",rotation=60,labelsize=8)
axs[1].tick_params(axis="y",labelsize=8)
fig.colorbar(im,ax=axs[1],shrink=.75);plt.show()
assert np.array_equal(probe_embedding(X),probe_embedding(p3))
assert np.array_equal(probe_embedding(p4),probe_embedding(p5))

## 8 · Compress execution behavior with PCA

One embedding might contain hundreds or thousands of observed output bits. PCA projects signatures into a smaller subspace while preserving directions of largest *variance*, not necessarily the distinctions most useful for solving a task.

We generate many random ASTs with maximum depth three, deduplicate **syntax**, and collect their behavior. Then we visualize the first two principal axes. Color shows operator count, *not a learned semantic label*.

**Predict:** Could two programs have near-identical 2D PCA coordinates but differ on a task-defining cell?

In [ ]:
UNARY=["RIGHT","LEFT","FLIP","NOT"]
BINARY=["OR","AND"]
def random_program(g,levels=3):
    if levels==0 or g.random()<.18:
        return X if g.random()<.87 else Z
    if g.random()<.72:
        return U(UNARY[int(g.integers(len(UNARY)))],random_program(g,levels-1))
    return B(BINARY[int(g.integers(len(BINARY)))],
             random_program(g,levels-1),random_program(g,levels-1))
g=np.random.default_rng(SEED+1)
programs=list(dict.fromkeys(sample+[B("OR",X,Z)]))
seen=set(programs)
for _ in range(25000):
    if len(programs)>=420:break
    p=random_program(g)
    if p not in seen:
        seen.add(p);programs.append(p)
assert len(programs)>=280
FEATURES_EXEC=np.vstack([probe_embedding(p) for p in programs])
def pca2(a):
    centered=np.asarray(a,dtype=float)-np.mean(a,axis=0)
    _,_,vt=np.linalg.svd(centered,full_matrices=False)
    return centered@vt[:2].T
coords=pca2(FEATURES_EXEC)
fig,ax=plt.subplots(figsize=(8.5,4))
scatter=ax.scatter(coords[:,0],coords[:,1],c=[size(p) for p in programs],cmap="viridis",
                   s=18,alpha=.7)
fig.colorbar(scatter,ax=ax,label="Number of AST nodes")
ax.set(xlabel="Principal component 1",ylabel="Principal component 2",
       title="PCA of *observed* execution signatures")
plt.show()
print("Unique syntax trees sampled:",len(programs),"probe output dimensions:",FEATURES_EXEC.shape[1])

## 9 · Split by semantic family before training an encoder

A random row split would leak closely related observations: the same program appears on multiple inputs, and different ASTs sometimes implement the **same function**. We first group programs by their complete 128-input truth table (possible only in this synthetic domain). We then split **semantic families**, not individual program rows, into train / validation / test.

This is stricter than necessary for some applications but gives a clean question: *does the model generalize to unseen functions expressed as unseen ASTs?*

For learned execution prediction, we also separate **training input masks** from **evaluation input masks**. The model will receive structure and input, then predict the output mask.

**Predict:** Why would a random split of program–input pairs be a misleading test of generalization to new programs?

In [ ]:
families=defaultdict(list)
for p in programs:
    key=signature(p,ALL_INPUTS).tobytes()
    families[key].append(p)
keys=list(families)
g=np.random.default_rng(SEED+2)
g.shuffle(keys)
n=len(keys);n_train=int(.7*n);n_val=int(.15*n)
split_families=[keys[:n_train],keys[n_train:n_train+n_val],keys[n_train+n_val:]]
train_programs=[p for key in split_families[0] for p in families[key]]
val_programs=[p for key in split_families[1] for p in families[key]]
test_programs=[p for key in split_families[2] for p in families[key]]
TRAIN_INPUT_IDS=np.array([0,1,2,4,8,16,32,64,127,21,37,90,55,103],dtype=int)
EVAL_INPUT_IDS=np.array([i for i in range(128) if i not in TRAIN_INPUT_IDS],dtype=int)
assert set(split_families[0]).isdisjoint(set(split_families[1]))
assert set(split_families[0]).isdisjoint(set(split_families[2]))
assert len(EVAL_INPUT_IDS)>0 and len(test_programs)>0
fig,ax=plt.subplots(figsize=(8.5,3.4))
ax.bar(["train","validation","test"],list(map(len,[train_programs,val_programs,test_programs])))
ax.set(ylabel="Programs (families disjoint)",title="Disjoint semantic-family split")
plt.show()
print("Semantic families:",n,"| program counts:",list(map(len,[train_programs,val_programs,test_programs])))
print("Training inputs:",len(TRAIN_INPUT_IDS),"| unseen evaluation inputs:",len(EVAL_INPUT_IDS))

## 10 · A fixed structural tensor for a small, bounded grammar

A model cannot ingest arbitrary Python objects. For this shallow demonstration we place one-hot operator IDs in the positions of a full binary tree, using heap indices: root 0, left child \(2i+1\), right child \(2i+2\).

This retains *where* an operation appears, unlike a bag-of-operators vector. The feature width grows exponentially with maximum depth; it is **not scalable to deep program graphs**. A recursive tree encoder, a DAG message-passing network, or a program transformer can avoid this fixed-size limitation. Our fixed feature map is intentionally simple so we can inspect every input to the neural model.

**Predict:** If a tree has one unary child, should it fill the left slot or both slots?

In [ ]:
MAX_DEPTH=3
TREE_SLOTS=2**(MAX_DEPTH+1)-1
def positional_features(p):
    out=np.zeros((TREE_SLOTS,len(OPS)),dtype=float)
    def visit(node,index):
        if index>=TREE_SLOTS:raise ValueError("Tree exceeds fixed feature depth")
        out[index,OP_INDEX[node.op]]=1.
        for j,c in enumerate(node.children):
            visit(c,2*index+1+j)
    visit(p,0)
    return out.ravel()
assert len(positional_features(X))==TREE_SLOTS*len(OPS)
assert not np.array_equal(positional_features(p1),positional_features(p2))
assert np.array_equal(bag(p1),bag(p2))
fig,axs=plt.subplots(1,2,figsize=(11,3.4),layout="constrained")
for ax,p,t in zip(axs,[p1,p2],["FLIP(RIGHT)","RIGHT(FLIP)"]):
    ax.imshow(positional_features(p).reshape(TREE_SLOTS,len(OPS))[:7],
              cmap="Blues",vmin=0,vmax=1,aspect="auto",interpolation="nearest")
    ax.set(xticks=range(len(OPS)),xticklabels=OPS,yticks=range(7),xlabel="Operation ID",
           ylabel="Binary-tree position",title=t)
    ax.tick_params(axis="x",rotation=45,labelsize=8)
plt.show()

## 11 · Train a neural emulator: AST + grid → predicted output grid

We now learn

\[\widehat{y}=f_\theta(\phi_{\rm position}(p),x),\]

using a tanh hidden layer and sigmoid output probabilities for each of seven binary cells. Training uses **binary cross-entropy**, with analytical backpropagation (as in lesson 00) and Adam updates implemented directly in NumPy.

The network's hidden activations form a learned **joint representation of program structure and input**. We can average activations across a fixed probe set to obtain a task-independent summary for a program, but that average is a defined pooling rule, not evidence of a perfect semantic representation.

**Predict:** Could a model achieve high average cell accuracy by predicting all zeros on a dataset with many mostly-empty outputs? What additional metrics would be useful?

In [ ]:
def make_rows(ps,grids):
    structural=np.array([positional_features(p) for p in ps])
    Xdata=np.concatenate([np.repeat(structural,len(grids),axis=0),
                          np.tile(grids.astype(float),(len(ps),1))],axis=1)
    Ydata=np.concatenate([execute(p,grids).astype(float) for p in ps],axis=0)
    return Xdata,Ydata

train_inputs=ALL_INPUTS[TRAIN_INPUT_IDS]
eval_inputs=ALL_INPUTS[EVAL_INPUT_IDS]
X_train,y_train=make_rows(train_programs,train_inputs)
X_val,y_val=make_rows(val_programs,train_inputs)
X_test,y_test=make_rows(test_programs,eval_inputs)

H=40
g=np.random.default_rng(SEED+3)
params={'W1':g.normal(0,.09,(X_train.shape[1],H)),
        'b1':np.zeros(H),
        'W2':g.normal(0,.09,(H,WIDTH)),
        'b2':np.zeros(WIDTH)}
def model_forward(p,X):
    hidden=np.tanh(X@p['W1']+p['b1'])
    logits=hidden@p['W2']+p['b2']
    predictions=1/(1+np.exp(-np.clip(logits,-35,35)))
    return predictions,hidden
def loss_and_grad(p,X,Y):
    prediction,hidden=model_forward(p,X)
    eps=1e-10
    loss=-np.mean(Y*np.log(np.clip(prediction,eps,1-eps))+
                 (1-Y)*np.log(np.clip(1-prediction,eps,1-eps)))
    dz=(prediction-Y)/(len(X)*WIDTH)
    grads={'W2':hidden.T@dz,'b2':dz.sum(axis=0)}
    dh=(dz@p['W2'].T)*(1-hidden**2)
    grads['W1']=X.T@dh;grads['b1']=dh.sum(axis=0)
    return float(loss),grads
print("Train program-grid rows:",X_train.shape,"| unseen program/unseen input rows:",X_test.shape)
assert X_train.shape[1]==TREE_SLOTS*len(OPS)+WIDTH
assert y_train.shape[1]==WIDTH

## 12 · Numerically verify the backward pass

Backpropagation is just the chain rule applied to the interpreter-emulator graph. A finite-difference check detects mistakes in derivatives or averaging conventions before training.

For parameter \(w\),

\[\frac{\partial\mathcal{L}}{\partial w}\approx
\frac{\mathcal{L}(w+\epsilon)-\mathcal{L}(w-\epsilon)}{2\epsilon}.\]

**Predict:** Why is a derivative check on the *neural emulator* a different test from checking that the *symbolic interpreter* produces correct grids?

In [ ]:
pcheck={k:v.copy() for k,v in params.items()}
_,analytic=loss_and_grad(pcheck,X_train[:10],y_train[:10])
errors=[]
for name,ix in [('W1',(0,0)),('W1',(8,9)),('b1',(7,)),
                ('W2',(4,2)),('b2',(3,))]:
    eps=1e-5
    pcheck[name][ix]+=eps
    plus=loss_and_grad(pcheck,X_train[:10],y_train[:10])[0]
    pcheck[name][ix]-=2*eps
    minus=loss_and_grad(pcheck,X_train[:10],y_train[:10])[0]
    pcheck[name][ix]+=eps
    errors.append(abs((plus-minus)/(2*eps)-analytic[name][ix]))
print("Maximum finite-difference gradient error:",max(errors))
assert max(errors)<1e-7

## 13 · Fit the emulator and inspect *both* program and input generalization

We optimize solely on training program families and training inputs. Validation uses different program families with the **same** training input patterns; the final diagnostic uses **different program families and new input patterns**. This is a deliberately hard extrapolation.

We plot BCE loss and distinguish **cell accuracy** from **exact-grid accuracy**. A program-grid row is correct only if *every cell* matches.

The goal is not to obtain a great score on this tiny architecture. It is to learn how to test whether a representation preserves the information needed to predict semantics.

**Predict:** If validation BCE decreases but exact-grid accuracy stays poor, what does that suggest about the quality of the learned representation for program search?

In [ ]:
def fit(p,X,Y,Xv,Yv,epochs=65,batch=256,lr=.012,seed=SEED):
    gen=np.random.default_rng(seed)
    m={k:np.zeros_like(w) for k,w in p.items()}
    v={k:np.zeros_like(w) for k,w in p.items()}
    history=[];step=0
    for epoch in range(epochs):
        perm=gen.permutation(len(X))
        for ind in np.array_split(perm,max(1,int(np.ceil(len(X)/batch)))):
            _,grad=loss_and_grad(p,X[ind],Y[ind]);step+=1
            for key in p:
                m[key]=.9*m[key]+.1*grad[key]
                v[key]=.999*v[key]+.001*grad[key]**2
                mh=m[key]/(1-.9**step)
                vh=v[key]/(1-.999**step)
                p[key]-=lr*mh/(np.sqrt(vh)+1e-8)
        history.append((loss_and_grad(p,X,Y)[0],loss_and_grad(p,Xv,Yv)[0]))
    return np.array(history)
history=fit(params,X_train,y_train,X_val,y_val)
test_pred,_=model_forward(params,X_test)
test_bits=test_pred>=.5
test_true=y_test.astype(bool)
test_cell_acc=float(np.mean(test_bits==test_true))
test_exact_acc=float(np.mean(np.all(test_bits==test_true,axis=1)))
fig,axs=plt.subplots(1,2,figsize=(11,3.6),layout="constrained")
axs[0].plot(history[:,0],label="train")
axs[0].plot(history[:,1],label="new program families / seen inputs")
axs[0].set(xlabel="Epoch",ylabel="Binary cross-entropy",title="Execution-emulator learning")
axs[0].legend(fontsize=8)
axs[1].bar(["cell","exact row"],[test_cell_acc,test_exact_acc])
axs[1].set(ylim=(0,1),ylabel="Accuracy",title="Unseen programs + unseen inputs")
plt.show()
print(f"Test cell accuracy={test_cell_acc:.3f}; exact-row accuracy={test_exact_acc:.3f}")
assert np.isfinite(history).all() and np.isfinite(test_pred).all()

## 14 · Inspect the learned model on a specific unseen program

The symbolic interpreter executes the rule exactly. The learned emulator approximates its output and can be wrong. Show a representative held-out program on several unseen inputs:

- input row,
- exact interpreter output,
- thresholded neural prediction,
- neural output probability.

This is why **neural search guidance must not replace exact execution checks**. A predicted semantic vector can prioritize candidates or suggest likely equivalences, but a solver should still verify candidates against every demonstration.

**Predict:** If the model mistakes even one required output cell, how should an exact program-synthesis evaluator treat that result?

In [ ]:
candidate=next((p for p in test_programs if size(p)>=3 and
                0 < execute(p,eval_inputs).mean() < 1),test_programs[0])
inputs=eval_inputs[:4]
XX,truth=make_rows([candidate],inputs)
confidence,_=model_forward(params,XX)
fig,axs=plt.subplots(4,4,figsize=(11,6.4),layout="constrained")
for j in range(4):
    for i,(data,title) in enumerate([(inputs[j],"Input"),
                                      (truth[j],"Interpreter"),
                                      (confidence[j]>=.5,"Neural threshold"),
                                      (confidence[j],"Neural probability")]):
        axs[j,i].imshow(np.asarray(data,float)[None,:],cmap="Greys",
                        vmin=0,vmax=1,aspect="auto",interpolation="nearest")
        axs[j,i].set(xticks=range(WIDTH),yticks=[])
        if j==0:axs[j,i].set_title(title)
plt.suptitle("Held-out program: "+pretty(candidate),fontsize=10)
plt.show()

## 15 · Hidden states as a learned *joint* representation

The tanh hidden vector is \(h_\theta(p,x)\), which depends on both a program tree and its current input. A program-level summary may pool hidden states over a fixed reference set:

\[\phi_\theta(p)=\frac1m\sum_{j=1}^m h_\theta(p,x_j).\]

The choice of reference inputs and pooling operation matters. In general, pooling loses information. A program that behaves differently on a rare input can look close to another program under a poorly chosen probe distribution.

We compare a 2D projection of *learned* pooled embeddings with PCA of the exact probe execution signatures. The plots are **qualitative representations**, not a proof of semantic retrieval quality.

**Predict:** Could an embedding be good for predicting one target but poor for finding functionally equivalent programs?

In [ ]:
visual_programs=programs[:140]
pooled=[]
for p in visual_programs:
    XX,_=make_rows([p],PROBES)
    _,hidden=model_forward(params,XX)
    pooled.append(hidden.mean(axis=0))
pooled=np.array(pooled)
exec_plot=np.array([probe_embedding(p) for p in visual_programs])
a=pca2(pooled);b=pca2(exec_plot)
fig,axs=plt.subplots(1,2,figsize=(11,3.8),layout="constrained")
for ax,pts,title in zip(axs,[a,b],["Learned pooled hidden states","Exact probe signatures"]):
    sc=ax.scatter(pts[:,0],pts[:,1],c=[size(p) for p in visual_programs],
                  s=16,alpha=.7,cmap="viridis")
    ax.set(xlabel="Component 1",ylabel="Component 2",title=title)
fig.colorbar(sc,ax=axs.ravel().tolist(),shrink=.8,label="AST node count")
plt.show()
print("Pooled representation dimensions:",pooled.shape[1])

## 16 · Retrieval is a testable downstream task

Suppose a search engine uses embedding similarity to propose reusable operations or candidate subprograms. We can compare:

- structural bag distance,
- exact execution disagreement on all inputs,
- and distance between learned pooled vectors.

Choose one query from a **held-out semantic family** and compare several candidate neighbors. Crucially, retrieval metrics should be computed on a *separate evaluation set* and matched against what search actually needs (exact solve discovery, cost, or semantic agreement), not just visually pleasing PCA clouds.

**Predict:** Can two programs be close under Euclidean distance yet differ on the single example relevant to the current task?

In [ ]:
query=test_programs[0]
candidate_pool=val_programs[:min(60,len(val_programs))]
def pooled_program(p):
    xx,_=make_rows([p],PROBES)
    return model_forward(params,xx)[1].mean(axis=0)
qvec=pooled_program(query)
embs=np.array([pooled_program(p) for p in candidate_pool])
dist=np.linalg.norm(embs-qvec,axis=1)
order=np.argsort(dist)[:6]
disagreement=np.array([np.mean(signature(query,ALL_INPUTS)!=signature(p,ALL_INPUTS))
                        for p in candidate_pool])
fig,axs=plt.subplots(1,2,figsize=(11,3.5),layout="constrained")
axs[0].scatter(dist,disagreement,alpha=.65,s=22)
axs[0].set(xlabel="Learned embedding distance",ylabel="Exact semantic Hamming disagreement",
           title="Does proximity predict behavioral similarity?")
axs[1].bar(range(len(order)),disagreement[order])
axs[1].set(xticks=range(len(order)),xticklabels=[f"#{i+1}" for i in range(len(order))],
           ylim=(0,1),xlabel="Nearest learned-vector neighbor",ylabel="Semantic disagreement",
           title="Inspect retrieved programs")
plt.show()
for rank,i in enumerate(order,1):
    print(rank,pretty(candidate_pool[i]),"disagreement",round(float(disagreement[i]),3))
assert np.all((disagreement>=0)&(disagreement<=1))

## 17 · Caching, shared subtrees, and stable identity

A tree may repeat the same subexpression. A directed acyclic graph (DAG) can store it once and let several later operations reuse it. Memoization caches an interpreter result keyed by the **exact program expression**, for the **same input batch**.

That cache key differs from a finite-probe semantic fingerprint. A fingerprint says two expressions produced the same results *on sampled probes*, not that they are universally equivalent. In the user's bundled-gene architecture, a physical gene may also contain multiple logical components; a stable gene ID plus component path prevents conflating them.

**Predict:** If we cache the output of RIGHT(INPUT) for one task, can we reuse the same tensor for an unrelated task with a different input?

In [ ]:
reused=U("RIGHT",X)
shared=B("OR",reused,U("FLIP",reused))
counter={"misses":0,"hits":0}
def execute_cached(expr,inputs,cache):
    if expr in cache:
        counter["hits"]+=1
        return cache[expr]
    counter["misses"]+=1
    if expr.op=="INPUT": out=np.asarray(inputs,dtype=bool).copy()
    elif expr.op=="ZERO": out=np.zeros_like(inputs,dtype=bool)
    else:
        # Compute children with memoization; evaluate this operator on already computed arrays.
        ch=[execute_cached(c,inputs,cache) for c in expr.children]
        if expr.op=="RIGHT":
            out=np.zeros_like(ch[0]);out[...,1:]=ch[0][...,:-1]
        elif expr.op=="LEFT":
            out=np.zeros_like(ch[0]);out[...,:-1]=ch[0][...,1:]
        elif expr.op=="FLIP": out=ch[0][...,::-1].copy()
        elif expr.op=="NOT": out=~ch[0]
        elif expr.op=="OR": out=ch[0]|ch[1]
        elif expr.op=="AND": out=ch[0]&ch[1]
        else: raise ValueError(expr.op)
    cache[expr]=out
    return out
batch=PROBES[:5]
cache={}
result=execute_cached(shared,batch,cache)
assert np.array_equal(result,execute(shared,batch))
assert counter["hits"]>0
print("Shared-subtree cache:",counter,"| physical cached AST entries:",len(cache))
fig,ax=plt.subplots(figsize=(7,3.2))
ax.bar(["computed subexpressions","cache hits"],[counter["misses"],counter["hits"]])
ax.set(ylabel="Count",title="Memoization reuses an identical expression on one batch")
plt.show()

## 18 · A realistic embedding interface for Grammar-UCT

Our toy features are deliberately inadequate for arbitrary ARC grids. A practical encoder would combine:

- **Task context:** input/output demonstrations, sizes, colors, object decompositions, and unsolved constraints.
- **Structure:** operator name, arity, source gene references, parameters, program depth, and component paths.
- **Execution:** generated masks or scalars across training examples, validity flags, transformations, and agreement with remaining target constraints.
- **Search context:** visited counts, duplicate/novelty signals, estimated compute cost, pruning state, and task/lineage provenance.

A task-conditioned policy could use these to rank **legal operation/source choices**, and a value network could predict expected future progress under a fixed budget/continuation policy. There is no guarantee that a good semantic embedding implies a good *decision* embedding; training objective and evaluation must be aligned with search.

**DAG encoder idea:** message passing over child-to-parent edges, learned aggregation, typed readouts, and caching of component embeddings; add permutation-invariant aggregation only for operators where child order does not matter. For variable-size outputs, use pooled sets/attention with carefully defined identity and provenance.

**Do not equate:** physical bundled gene ≠ each component ≠ AST node ≠ semantic equivalence class. Those are four distinct entities.


## 19 · Failure analysis: what the representations *do not* prove

| Representation | Preserves | Loses or risks |
|---|---|---|
| Bag of operators | Multiset counts | Order, parentage, arity, source identity |
| Random recursive tree embedding | Some tree structure and child position | No learned semantic alignment; collisions/compression possible |
| Fixed-position tensor | Node labels at bounded tree positions | Exponential maximum-depth width; cannot naturally scale |
| Finite-probe execution signature | Exact values on chosen inputs | Differences outside probes; input-distribution dependence |
| PCA of executions | Large-variance directions | Rare important behavior and exact equivalence |
| Neural joint embedding | What supervised prediction encourages | Out-of-distribution failures; learned shortcuts |

Our 128-input exhaustive oracle establishes functional equivalence only for the **toy width-seven Boolean domain**. A high emulator cell accuracy does not establish reliable program synthesis. A useful real benchmark must assess held-out ARC tasks, program composition lengths, unseen operations/objects, cache correctness, wall-time budgets, and exact successful program discovery.

Remember lesson 03: uncertainty over *found* hypotheses does not account for missing programs. The same caution applies to vector neighborhoods: the nearest embedding neighbor is not necessarily a good hypothesis.

## 20 · Exercises — change one thing at a time

1. **Syntax equivalence:** prove from interpreter semantics that FLIP(FLIP(x)) equals INPUT for every width-seven Boolean row. Why might their recursive embeddings still differ?
2. **Composition:** show a particular input where FLIP(RIGHT(x)) differs from RIGHT(FLIP(x)). Explain the boundary information loss.
3. **Operator invariance:** set the recursive encoder's left/right matrices equal and compare OR(a,b) to OR(b,a). Which other operator types would be incorrectly treated as commutative if the grammar were expanded?
4. **Probe failure:** find another pair of distinct programs equal on ALL_INPUTS[[0]] but different on the full universe.
5. **More expressive features:** replace fixed positional features with a learned recursive encoder. Describe how you would propagate output-loss gradients through shared subtrees and handle variable arity.
6. **New semantic families:** change the family split from truth tables to exact AST identity. What leak reappears, and what is the appropriate claim for that evaluation?
7. **Metric evaluation:** compute Spearman correlation between neural embedding distance and full-domain semantic Hamming distance for held-out query/candidate pairs. Does a high correlation guarantee exact nearest-neighbor retrieval?
8. **New probe distribution:** replace singleton-heavy PROBES with mostly empty inputs; identify any additional signature collisions.
9. **DAG caching:** compute cached expressions for two different input batches. Modify the key so stale tensors cannot be reused inadvertently.
10. **ARC integration:** design a stable component-level embedding cache key using physical gene ID, component path, task context, encoder version, and any necessary execution/probe versioning.

Try the experiments before reading the answers.

## 21 · Worked solutions and reconstruction

1. Reversal is an involution, so applying it twice returns the original row. The recursive encoder has no algebraic equality constraints and sees a different tree.
2. Place a true cell at an edge: shifting before reversing loses the opposite boundary compared with reversing before shifting.
3. With identical child matrices the OR child swap contributes the same sum. Order-sensitive operations (e.g., subtraction, string concatenation, crop-then-shift) would lose important information if treated the same way.
4. RIGHT(INPUT) and LEFT(INPUT) both produce zero on an all-zero input, but disagree on a singleton boundary input.
5. Use recursive differentiation or autodiff through the child states, accumulating parameter gradients for reused shared operation matrices. Shared DAG nodes require correct accumulation from every parent.
6. Semantically equivalent programs can cross splits even if ASTs do not. The result then measures generalization to new syntax, not unseen functions.
7. Correlation describes broad ranking alignment, not whether the top neighbor satisfies every task constraint. Confirm exact replay.
8. Sparse/zero-only probes cannot distinguish many operations; compare all-pairs equality of signatures to full-domain equality.
9. Include an immutable digest of the input batch or scope the cache to one task/input batch, and avoid mutating cached results.
10. Include stable task or dataset version, physical gene ID, component path, encoder/checkpoint version, and probe or source-grid version. Caching across tasks requires input-aware semantics.

### Reconstruct without looking
Implement an AST interpreter, operator-bag baseline, recursive encoder, full-domain truth-table tests (small domain), probe signature, 2D PCA plot, supervised AST+input emulator with finite-difference gradient check, task/program-disjoint evaluation, and a memoized DAG interpreter.

### Primary reading and intellectual lineage
- Tai, Socher & Manning (2015), *Improved Semantic Representations From Tree-Structured Long Short-Term Memory Networks*.
- Allamanis et al. (2018), *Learning to Represent Programs with Graphs*.
- Li et al. (2015), *Gated Graph Sequence Neural Networks*.
- Vinyals et al. (2015), *Order Matters: Sequence to Sequence for Sets* (order/invariance).
- Ellis et al. (2021), *DreamCoder: Bootstrapping Inductive Program Synthesis with Wake-Sleep Library Learning*.

**Next: 07 · Learning reusable abstractions.** We'll ask how frequently reused subtrees can become library primitives without hiding library-definition complexity or leaking information from evaluation tasks.

**Scope:** original self-contained instructional experiment. The untrained recursive encoder and trained positional emulator are intentionally distinguished. No claim is made that the learned vectors are universally semantic or ARC-competitive.